# Plotting Precipitation as Time Series

Now, we will test how different ICON files can be combined together. This is done with `2d_cloud` files and a total (domain-average) precipitation time series will be plotted at the end. 

## Load Python Libraries 

In [ ]:
%matplotlib inline

# system libs
import os, sys, glob
import datetime

# array operators and netcdf datasets
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

# plotting
import pylab as plt
import seaborn as sns
sns.set_context('talk')

import matplotlib.dates as mdates
myFmt = mdates.DateFormatter('%H:%M')


## Open Data with `xarray`

### General Paths

In [ ]:
exercise_path = '/home/b/b383413/workspace'
data_path = f'{exercise_path}/data'
icon_data_path = f'{data_path}/icon'

This is very similar to the tasks already done.

In [ ]:
os.listdir(icon_data_path)

Now, we select the `experiment`folder in which pre-calculated experiments are stored. **Later, you need to change this directory to your ICON output directory!** 

In [ ]:
exp_name = 'cesar1-20240806-exp001testothertime'
exp_path = f'/home/b/b383413/workspace/icon-build/experiments/{exp_name}'

### Base Files 

In [ ]:
#dset = xr.open_mfdataset( f'{exp_path}/2d_cloud_DOM01_ML_20210516T*Z.nc', chunks={'time':1}, combine='by_coords' ) 
dset = xr.open_mfdataset( f'{exp_path}/2d_cloud_DOM01_ML_20240806T*Z.nc', chunks={'time':1}, combine='by_coords' ) 


This is the content of the dataset. 

In [ ]:
dset

Now, the time dimension has a length of `288`, i.e. a full day with output each 5 minutes is accumulated. The full data are not loaded into memory so far. We use the capabilities of the `dask` module to make out-of-memory computations ... very convenient! 

### Adding Grid-Scale and Sub-Gridscale Rain

In [ ]:
rain = dset['rain_con_rate'] + dset['rain_gsp_rate']
rain.attrs['long_name'] = 'rain rate'

TODO for later: 
* Is it OK to neglect graupel / snow & hail?

### Calculate Average Time Series

In [ ]:
rain_mean = rain.mean( 'ncells' )

This is still out-of-memory... But, now we do the calculations!

In [ ]:
rain_mean = rain_mean.compute()

## Plot the Time Series

### Simple (naive) Plotting 

Plotting is now not complicated. We just use the capabilities of `xarray`.

In [ ]:
fig = plt.figure( figsize = (14,6))
rain_mean.plot( lw = 3 )
sns.despine()

Do you understand this figure? Hmm, time series plotting is perhaps not so simple...

We need to do two things:
* convert the rain unit into mm per day
* get an understandable time axis.

### Unit Conversion

OK, we see the unit kg m-2 s-1. A day has 3600*24 seconds. One liter of water weighs one kilogram. This liter of water put on one square meter has a column height of 1 mm. OK, so far?



In [ ]:
conversion_factor = 3600 * 24

In [ ]:
rain_mean = conversion_factor * rain_mean
rain_mean.attrs['units'] = 'mm day-1'

### Time Axis 

In [ ]:
rain_mean.time

The time format is special: `%Y%m%d.%f`. We use the `datetime` module to convert the time vector. We will use two function.

In [ ]:

def convert_time(t, roundTo = 60.):

    '''
    Utility converts between two time formats A->B or B->A: 
    
    Parameters
    ----------
    t : float or datetime object
        time 
        A = either float as %Y%m%d.%f where %f is fraction of the day
        B = datetime object
    Returns
    -------
    tout : datetime object or float
        time, counterpart to t
    '''

    t0 = datetime.datetime(1970, 1, 1)

    if type(t) == type(t0):

        tout = np.int( t.strftime('%Y%m%d') )
        date = datetime.datetime.strptime(str(tout), '%Y%m%d')

        dt = (t - date).total_seconds()
        frac =  dt / (24 * 3600)

        tout +=  frac
    else:
        
        date = np.int(t)
        frac = t - date

        tout = datetime.datetime.strptime(str(date), '%Y%m%d')
        tout += datetime.timedelta( days = frac )

    return tout 


######################################################################
######################################################################

def convert_timevec( timevec ):
    '''
    Utility converts between an array of two time formats A->B or B->A: 
    
    Parameters
    ----------
    timevec : list or array
        time list
        A = either float as %Y%m%d.%f where %f is fraction of the day
        B = datetime object
    Returns
    -------
    tout : list
        list of datetime objects or floats
        time, counterpart to t
    '''
    
    times = []
    for tfloat in timevec:
        times += [ convert_time( tfloat ), ]
    
    times = np.array( times )
    
    return times


The above functions is now applied to the `time` coordinate:

In [ ]:
rain_mean['time'] = convert_timevec( rain_mean.time.data )

We will need the function `convert_timevec` rather often. Therefore, the two function are included in an `analysis/tools.py` module which can be loaded via:

```python
sys.path.append( '/work/bb1224/2024_MS-COURSE/tools/analysis' )
from tools import convert_timevec
```

In [ ]:
fig = plt.figure( figsize = (14,6))

plt.gca().xaxis.set_major_formatter(myFmt)
rain_mean.plot( lw = 3 )
sns.despine()

## Tasks

Use this notebook to explore the time series of other variables.

* How do different cloud cover variables look like?

* How does LWP evolve? (variable `tqc_dia`)

* How does CAPE look like?